In [1]:
## Load the raw datasets and create working copies

import pandas as pd
from pathlib import Path

project_path = Path.home() / "Desktop" / "Supply Chain Disruption Intelligence"
raw_path = project_path / "Data" / "raw"

customers_clean = pd.read_csv(raw_path / "customer_master.csv")
products_clean = pd.read_csv(raw_path / "product_master.csv")
suppliers_clean = pd.read_csv(raw_path / "supplier_master.csv")
sales_clean = pd.read_csv(raw_path / "sales_orders.csv")
procurement_clean = pd.read_csv(raw_path / "procurement_orders.csv")


In [2]:
## Standardize column names

clean_datasets = {
    "customers": customers_clean,
    "products": products_clean,
    "suppliers": suppliers_clean,
    "sales": sales_clean,
    "procurement": procurement_clean
}

for name, df in clean_datasets.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )

    print(f"{name}: {df.columns.tolist()}")

customers: ['customer_id', 'customer_name', 'industry', 'market_segment', 'country', 'city', 'latitude', 'longitude']
products: ['product_id', 'sku', 'product_name', 'category', 'subcategory', 'unit', 'unit_cost', 'standard_price', 'launch_date', 'discontinuation_date']
suppliers: ['supplier_id', 'supplier_name', 'country', 'region', 'on_time_delivery_rate', 'certification_level', 'preferred_supplier_flag']
sales: ['order_id', 'customer_id', 'product_id', 'order_date', 'order_status', 'order_quantity', 'unit_price', 'discount', 'shipping_mode', 'shipping_carrier', 'shipping_date_scheduled', 'shipping_date_actual', 'delivery_status', 'late_delivery_risk_flag', 'vat_rate', 'cogs', 'unit_price_effective', 'order_total', 'vat_amount', 'profit_per_order']
procurement: ['po_id', 'supplier_id', 'raw_material_id', 'order_date', 'order_quantity', 'unit_cost', 'delivery_date_planned', 'delivery_date_actual', 'total_cost']


In [3]:
## Convert date columns to proper datetime format

date_columns = {
    "products": ["launch_date", "discontinuation_date"],
    "sales": [
        "order_date",
        "shipping_date_scheduled",
        "shipping_date_actual"
    ],
    "procurement": [
        "order_date",
        "delivery_date_planned",
        "delivery_date_actual"
    ]
}

for name, columns in date_columns.items():
    df = clean_datasets[name]

    for col in columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    print(f"{name}: date conversion complete")

products: date conversion complete
sales: date conversion complete
procurement: date conversion complete


In [4]:
## Check for invalid dates

for name, columns in date_columns.items():
    df = clean_datasets[name]

    for col in columns:
        invalid = df[col].isna().sum()
        print(f"{name}.{col}: {invalid} missing/invalid dates")

products.launch_date: 0 missing/invalid dates
products.discontinuation_date: 200 missing/invalid dates
sales.order_date: 0 missing/invalid dates
sales.shipping_date_scheduled: 0 missing/invalid dates
sales.shipping_date_actual: 0 missing/invalid dates
procurement.order_date: 0 missing/invalid dates
procurement.delivery_date_planned: 0 missing/invalid dates
procurement.delivery_date_actual: 0 missing/invalid dates


In [5]:
## Check numeric columns for impossible values

# Check for non-positive quantities and negative costs/prices
checks = {
    "products.unit_cost": (products_clean, "unit_cost", "negative"),
    "products.standard_price": (products_clean, "standard_price", "negative"),
    "sales.order_quantity": (sales_clean, "order_quantity", "non_positive"),
    "sales.unit_price": (sales_clean, "unit_price", "negative"),
    "sales.discount": (sales_clean, "discount", "outside_0_1"),
    "procurement.order_quantity": (procurement_clean, "order_quantity", "non_positive"),
    "procurement.unit_cost": (procurement_clean, "unit_cost", "negative"),
    "procurement.total_cost": (procurement_clean, "total_cost", "negative")
}

for label, (df, col, rule) in checks.items():
    if rule == "negative":
        count = (df[col] < 0).sum()
    elif rule == "non_positive":
        count = (df[col] <= 0).sum()
    else:
        count = (~df[col].between(0, 1)).sum()

    print(f"{label}: {count} values violating rule")

products.unit_cost: 0 values violating rule
products.standard_price: 0 values violating rule
sales.order_quantity: 0 values violating rule
sales.unit_price: 0 values violating rule
sales.discount: 0 values violating rule
procurement.order_quantity: 0 values violating rule
procurement.unit_cost: 0 values violating rule
procurement.total_cost: 0 values violating rule


In [6]:
## Validate procurement cost calculations

procurement_clean["calculated_cost"] = (
    procurement_clean["order_quantity"]
    * procurement_clean["unit_cost"]
)

cost_mismatch = (
    procurement_clean["total_cost"] - procurement_clean["calculated_cost"]
).abs() > 0.01

print("Cost mismatches:", cost_mismatch.sum())

Cost mismatches: 0


In [7]:
## Save the cleaned datasets

procurement_clean.drop(columns=["calculated_cost"], inplace=True)

In [ ]:
## Save the cleaned datasets

processed_path = project_path / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

for name, df in clean_datasets.items():
    df.to_csv(processed_path / f"{name}.csv", index=False)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!


In [9]:
## Quick verification

for name in clean_datasets:
    file_path = processed_path / f"{name}.csv"
    print(f"{name}: {'Saved' if file_path.exists() else 'Missing'}")

customers: Saved
products: Saved
suppliers: Saved
sales: Saved
procurement: Saved
